# 🏗️ Sprint 1 — Foundation & Input Pipeline
**Project:** Autonomous Company Research & Report Generation Agent  
**Module:** 3 | **Day:** 1 of 5 | **Sprint Goal:** Input pipeline working end-to-end

## What we build today
```
Company Name Input → Validate → Normalise → Acknowledge → Ready for Sprint 2
```

## Sprint 1 User Story
> **US-01:** As an analyst, I can submit a company name and receive a structured ticket confirmation so I know the research has been initiated.

**Definition of Done:** Function returns a structured ticket dict within 1 second for any valid company name input.

## 1. Install Dependencies

In [1]:
# Install all dependencies for the full project (run once)
%pip install -q \
    openai \
    langchain \
    langchain-openai \
    langchain-core \
    langgraph \
    pinecone-client \
    python-dotenv \
    tiktoken \
    rich

Note: you may need to restart the kernel to use updated packages.


## 2. Environment Setup

In [2]:
import os
from dotenv import load_dotenv

# Load from .env file if present
load_dotenv()

# ── Set your keys here if not using .env ─────────────────────────────────────
# Uncomment and fill in if you are NOT using a .env file:
# os.environ["OPENAI_API_KEY"]   = "sk-..."
# os.environ["PINECONE_API_KEY"] = "pcsk_..."

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY", "")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")

# ── Validate ──────────────────────────────────────────────────────────────────
assert OPENAI_API_KEY,   "❌ OPENAI_API_KEY not set. Add it above or create a .env file."
assert PINECONE_API_KEY, "❌ PINECONE_API_KEY not set. Add it above or create a .env file."

print("✅ OpenAI key loaded   :", OPENAI_API_KEY[:12] + "...")
print("✅ Pinecone key loaded :", PINECONE_API_KEY[:12] + "...")

✅ OpenAI key loaded   : OPENAI...
✅ Pinecone key loaded : pcsk_eUK55_T...


## 3. Agent State Schema

This `TypedDict` is the **single source of truth** shared across every node in the LangGraph agent.
Every sprint adds new fields to this schema — we define it fully here so all notebooks are compatible.

In [3]:
from typing import TypedDict, Optional, List, Dict, Any

class AgentState(TypedDict):
    """Shared state passed between every LangGraph node."""

    # ── Sprint 1: Input ───────────────────────────────────────────────────────
    company_name:    str                        # Raw company name from analyst
    ticket_id:       str                        # Unique research ticket ID
    initiated_at:    str                        # ISO timestamp
    status:          str                        # Current pipeline status
    errors:          List[str]                  # Accumulated error messages

    # ── Sprint 2: Research ────────────────────────────────────────────────────
    raw_research:    Optional[str]              # Raw text from OpenAI web search
    research_chunks: Optional[List[str]]        # Text split into chunks
    pinecone_ids:    Optional[List[str]]        # IDs of upserted vectors

    # ── Sprint 3: RAG + Agent ─────────────────────────────────────────────────
    retrieved_chunks: Optional[List[str]]       # Top chunks from Pinecone
    reranked_chunks:  Optional[List[str]]       # Top-3 after Cohere reranking
    risk_score:       Optional[str]             # low / medium / high
    opportunity_score: Optional[int]            # 1–10
    retry_count:      int                       # For retry logic

    # ── Sprint 4: Report ──────────────────────────────────────────────────────
    report_sections:  Optional[Dict[str, str]]  # 6 sections keyed by name
    notion_url:       Optional[str]             # URL of created Notion page
    report_ready:     bool                      # True when report is complete

    # ── Audit trail ───────────────────────────────────────────────────────────
    workflow_path:    List[str]                 # Names of nodes visited

print("✅ AgentState schema defined — ", len(AgentState.__annotations__), "fields")

✅ AgentState schema defined —  17 fields


## 4. Input Validation Node

In [4]:
import uuid
from datetime import datetime, timezone

def validate_input_node(state: AgentState) -> AgentState:
    """
    Node 1: Validate and normalise the company name input.

    Reads:  state['company_name']
    Writes: state['company_name'] (cleaned), state['ticket_id'],
            state['initiated_at'], state['status'], state['errors']
    """
    print("[VALIDATE] Validating input...")

    raw = state.get("company_name", "").strip()
    errors = list(state.get("errors", []))

    # ── Validation rules ──────────────────────────────────────────────────────
    if not raw:
        errors.append("Company name is empty.")
        return {
            **state,
            "status": "error_invalid_input",
            "errors": errors,
            "workflow_path": state.get("workflow_path", []) + ["validate_input"]
        }

    if len(raw) < 2:
        errors.append(f"Company name too short: '{raw}'")
        return {
            **state,
            "status": "error_invalid_input",
            "errors": errors,
            "workflow_path": state.get("workflow_path", []) + ["validate_input"]
        }

    if len(raw) > 200:
        errors.append("Company name too long (max 200 chars).")
        return {
            **state,
            "status": "error_invalid_input",
            "errors": errors,
            "workflow_path": state.get("workflow_path", []) + ["validate_input"]
        }

    # ── Normalise ─────────────────────────────────────────────────────────────
    # Remove extra whitespace, title-case the name
    cleaned = " ".join(raw.split())

    ticket_id    = f"RES-{datetime.now(timezone.utc).strftime('%Y%m%d')}-{str(uuid.uuid4())[:8].upper()}"
    initiated_at = datetime.now(timezone.utc).isoformat()

    print(f"[VALIDATE] ✅ Company: '{cleaned}' | Ticket: {ticket_id}")

    return {
        **state,
        "company_name":  cleaned,
        "ticket_id":     ticket_id,
        "initiated_at":  initiated_at,
        "status":        "validated",
        "errors":        errors,
        "workflow_path": state.get("workflow_path", []) + ["validate_input"]
    }

print("✅ validate_input_node defined")

✅ validate_input_node defined


## 5. Acknowledge Node

In [5]:
def acknowledge_node(state: AgentState) -> AgentState:
    """
    Node 2: Generate a confirmation message for the analyst.
    In production this would send a Telegram message.
    In the notebook, it prints a formatted confirmation.

    Reads:  state['company_name'], state['ticket_id'], state['initiated_at']
    Writes: state['status']
    """
    print("[ACKNOWLEDGE] Sending confirmation...")

    confirmation = (
        f"\n{'='*55}\n"
        f"  ✅ RESEARCH INITIATED\n"
        f"{'='*55}\n"
        f"  🎫 Ticket ID   : {state['ticket_id']}\n"
        f"  🏢 Company     : {state['company_name']}\n"
        f"  🕐 Started at  : {state['initiated_at']}\n"
        f"  📊 Status      : Research pipeline starting...\n"
        f"{'='*55}"
    )
    print(confirmation)

    return {
        **state,
        "status": "acknowledged",
        "workflow_path": state.get("workflow_path", []) + ["acknowledge"]
    }

print("✅ acknowledge_node defined")

✅ acknowledge_node defined


## 6. Routing Logic

In [6]:
from langgraph.graph import END

def route_after_validation(state: AgentState) -> str:
    """
    Conditional edge: after validate_input_node decide next step.
    Returns node name string.
    """
    if state["status"] == "error_invalid_input":
        print(f"[ROUTER] Invalid input → END | Errors: {state['errors']}")
        return "error_handler"
    return "acknowledge"


def error_handler_node(state: AgentState) -> AgentState:
    """Terminal node for validation errors."""
    print("[ERROR HANDLER] Pipeline stopped.")
    for err in state.get("errors", []):
        print(f"  ❌ {err}")
    return {
        **state,
        "status": "failed",
        "workflow_path": state.get("workflow_path", []) + ["error_handler"]
    }

print("✅ Routing logic defined")

c:\Users\abiod\IRONHACK PROJECT\Research_and_Report_Generation_Agent\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


✅ Routing logic defined


## 7. Build Sprint 1 Graph

In [7]:
from langgraph.graph import StateGraph, END

def build_sprint1_graph():
    """Build and compile the Sprint 1 LangGraph."""
    graph = StateGraph(AgentState)

    # ── Register nodes ────────────────────────────────────────────────────────
    graph.add_node("validate_input", validate_input_node)
    graph.add_node("acknowledge",    acknowledge_node)
    graph.add_node("error_handler",  error_handler_node)

    # ── Entry point ───────────────────────────────────────────────────────────
    graph.set_entry_point("validate_input")

    # ── Edges ─────────────────────────────────────────────────────────────────
    graph.add_conditional_edges(
        "validate_input",
        route_after_validation,
        { "acknowledge": "acknowledge", "error_handler": "error_handler" }
    )
    graph.add_edge("acknowledge",   END)
    graph.add_edge("error_handler", END)

    return graph.compile()


sprint1_graph = build_sprint1_graph()
print("✅ Sprint 1 graph compiled")
print("   Flow: validate_input → [acknowledge | error_handler] → END")

✅ Sprint 1 graph compiled
   Flow: validate_input → [acknowledge | error_handler] → END


## 8. Helper — Create Initial State

In [8]:
def create_initial_state(company_name: str) -> AgentState:
    """Create a clean initial AgentState for a new research request."""
    return AgentState(
        company_name     = company_name,
        ticket_id        = "",
        initiated_at     = "",
        status           = "pending",
        errors           = [],
        raw_research     = None,
        research_chunks  = None,
        pinecone_ids     = None,
        retrieved_chunks = None,
        reranked_chunks  = None,
        risk_score       = None,
        opportunity_score = None,
        retry_count      = 0,
        report_sections  = None,
        notion_url       = None,
        report_ready     = False,
        workflow_path    = []
    )

print("✅ create_initial_state helper defined")

✅ create_initial_state helper defined


## 9. Run Tests — Sprint 1 DoD Verification

In [9]:
# ── Test 1: Valid company name ────────────────────────────────────────────────
print("\n" + "#"*60)
print("TEST 1 — Valid company name: 'Anthropic'")
print("#"*60)

state1 = create_initial_state("Anthropic")
result1 = sprint1_graph.invoke(state1)

print(f"\nStatus        : {result1['status']}")
print(f"Ticket ID     : {result1['ticket_id']}")
print(f"Company       : {result1['company_name']}")
print(f"Workflow path : {' → '.join(result1['workflow_path'])}")

assert result1["status"] == "acknowledged",  "❌ Expected status: acknowledged"
assert result1["ticket_id"].startswith("RES-"), "❌ Expected ticket ID to start with RES-"
assert result1["company_name"] == "Anthropic", "❌ Expected company name: Anthropic"
print("\n✅ TEST 1 PASSED")


############################################################
TEST 1 — Valid company name: 'Anthropic'
############################################################
[VALIDATE] Validating input...
[VALIDATE] ✅ Company: 'Anthropic' | Ticket: RES-20260518-3008AFB1
[ACKNOWLEDGE] Sending confirmation...

  ✅ RESEARCH INITIATED
  🎫 Ticket ID   : RES-20260518-3008AFB1
  🏢 Company     : Anthropic
  🕐 Started at  : 2026-05-18T16:55:04.150183+00:00
  📊 Status      : Research pipeline starting...

Status        : acknowledged
Ticket ID     : RES-20260518-3008AFB1
Company       : Anthropic
Workflow path : validate_input → acknowledge

✅ TEST 1 PASSED


In [10]:
# ── Test 2: Empty input ───────────────────────────────────────────────────────
print("\n" + "#"*60)
print("TEST 2 — Empty company name")
print("#"*60)

state2  = create_initial_state("")
result2 = sprint1_graph.invoke(state2)

print(f"Status        : {result2['status']}")
print(f"Errors        : {result2['errors']}")
print(f"Workflow path : {' → '.join(result2['workflow_path'])}")

assert result2["status"] == "failed", "❌ Expected status: failed"
assert len(result2["errors"]) > 0,    "❌ Expected at least one error"
print("\n✅ TEST 2 PASSED")


############################################################
TEST 2 — Empty company name
############################################################
[VALIDATE] Validating input...
[ROUTER] Invalid input → END | Errors: ['Company name is empty.']
[ERROR HANDLER] Pipeline stopped.
  ❌ Company name is empty.
Status        : failed
Errors        : ['Company name is empty.']
Workflow path : validate_input → error_handler

✅ TEST 2 PASSED


In [11]:
# ── Test 3: Messy input (extra spaces) ───────────────────────────────────────
print("\n" + "#"*60)
print("TEST 3 — Messy input: '  openAI   '")
print("#"*60)

state3  = create_initial_state("  openAI   ")
result3 = sprint1_graph.invoke(state3)

print(f"Status        : {result3['status']}")
print(f"Cleaned name  : '{result3['company_name']}'")

assert result3["status"] == "acknowledged",  "❌ Expected acknowledged"
assert result3["company_name"] == "openAI",  "❌ Expected cleaned name: 'openAI'"
print("\n✅ TEST 3 PASSED")


############################################################
TEST 3 — Messy input: '  openAI   '
############################################################
[VALIDATE] Validating input...
[VALIDATE] ✅ Company: 'openAI' | Ticket: RES-20260518-0EB8EC19
[ACKNOWLEDGE] Sending confirmation...

  ✅ RESEARCH INITIATED
  🎫 Ticket ID   : RES-20260518-0EB8EC19
  🏢 Company     : openAI
  🕐 Started at  : 2026-05-18T16:55:04.225138+00:00
  📊 Status      : Research pipeline starting...
Status        : acknowledged
Cleaned name  : 'openAI'

✅ TEST 3 PASSED


In [12]:
# ── Sprint 1 DoD Summary ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("SPRINT 1 — DEFINITION OF DONE CHECK")
print("="*60)
print("✅ Valid company name → ticket created with RES- prefix")
print("✅ Empty input → routed to error handler, pipeline stops")
print("✅ Messy input → cleaned before processing")
print("✅ workflow_path audit trail populated on every run")
print("✅ AgentState schema defined and compatible with Sprints 2–5")
print("\n🎉 Sprint 1 DoD: MET — Ready for Sprint 2")


SPRINT 1 — DEFINITION OF DONE CHECK
✅ Valid company name → ticket created with RES- prefix
✅ Empty input → routed to error handler, pipeline stops
✅ Messy input → cleaned before processing
✅ workflow_path audit trail populated on every run
✅ AgentState schema defined and compatible with Sprints 2–5

🎉 Sprint 1 DoD: MET — Ready for Sprint 2


## 10. Save State for Sprint 2

We export `create_initial_state` and `AgentState` so Sprint 2 can import them.

In [13]:
import json, pathlib

# Save a sample final state as JSON for inspection / handoff
sample = dict(result1)
pathlib.Path("sprint1_output.json").write_text(json.dumps(sample, indent=2))
print("✅ Sprint 1 output saved to sprint1_output.json")
print("   Load in Sprint 2 with: json.load(open('sprint1_output.json'))")

✅ Sprint 1 output saved to sprint1_output.json
   Load in Sprint 2 with: json.load(open('sprint1_output.json'))
